# Synthetic Pairs Better - Priority-Weighted Version

## Key Features
This version uses **priority-weighted sampling** to emphasize important error types:

1. **HIGH priority (kept more often)**: â↔a, î↔i, i↔ı, e↔a, e↔i
2. **MEDIUM priority**: Other diacritics, vowels, consonants  
3. **LOW priority (fixed more often)**: Case changes, punctuation, spacing

## Why This Matters
OCR post-correction models often waste capacity learning trivial fixes (case, punctuation).
By weighting synthetic data generation, we ensure the model sees more examples of the 
important error types that actually affect text quality.

## Output
- `df_synthetic_better_random.pkl` - Priority-weighted synthetic pairs
- New column: `high_priority_remaining` - Count of high-priority errors per row

In [ ]:
import pandas as pd
import numpy as np
import random
import re
import json
from difflib import SequenceMatcher

random.seed(42)
np.random.seed(42)

# Load data - MUST have Better_Pred and Better_Ref columns
# Using df_tagged_better.pkl created in re-alignment.ipynb
df_tagged = pd.read_pickle('df_tagged_better.pkl')

print(f'df_tagged: {len(df_tagged)} rows')
print(f'Has Better_Pred: {"Better_Pred" in df_tagged.columns}')
print(f'Has Better_Ref: {"Better_Ref" in df_tagged.columns}')

if 'Better_Pred' not in df_tagged.columns:
    raise ValueError('Better_Pred column not found! Run re-alignment.ipynb first.')

# Load split from the SAME source as synthethic_pairs.ipynb
with open('byt5_ungated_baseline/split_info.json', 'r') as f:
    split_info = json.load(f)
train_indices = set(split_info['train_indices'])
dev_indices = set(split_info['dev_indices'])
test_indices = set(split_info['test_indices'])
print(f'Split: train={len(train_indices)}, dev={len(dev_indices)}, test={len(test_indices)}')

In [ ]:
# =============================================================================
# ERROR PRIORITY CLASSIFICATION
# =============================================================================

# TOP PRIORITY PAIRS - specific error patterns to emphasize (both directions)
TOP_PRIORITY_PAIRS = {
    ('â', 'a'), ('a', 'â'),  # circumflex a
    ('î', 'i'), ('i', 'î'),  # circumflex i
    ('i', 'ı'), ('ı', 'i'),  # dotted/dotless i
    ('e', 'a'), ('a', 'e'),  # visual similarity
    ('e', 'i'), ('i', 'e'),  # vowel confusion
    # Uppercase versions
    ('Â', 'A'), ('A', 'Â'),
    ('Î', 'İ'), ('İ', 'Î'),
    ('İ', 'I'), ('I', 'İ'),
    ('E', 'A'), ('A', 'E'),
    ('E', 'İ'), ('İ', 'E'),
}

# Full Turkish diacritic character set
DIACRITIC_CHARS = set(['Â', 'Ç', 'Î', 'Ö', 'Û', 'Ü', 'â', 'ç', 'î', 'ö', 'û', 'ü', 'ğ', 'İ', 'ı', 'Ş', 'ş', 'Ğ'])

# All vowels (for vowel confusion detection)
VOWELS = set('aeiouAEIOUâîûöüÂÎÛÖÜıİ')

# Low priority characters
LOW_PRIORITY_CHARS = set(' \t\n.,;:!?()[]"\'\'"-–—/')

# Weights for fixing errors (higher = more likely to fix = less likely to remain in training)
PRIORITY_FIX_WEIGHTS = {
    'high': 0.3,    # Keep high priority errors (less likely to fix)
    'medium': 1.0,  # Neutral
    'low': 3.0,     # Fix low priority errors (more likely to fix)
}

def classify_diff_priority(diff):
    """
    Classify a diff by priority based on the characters involved.
    Returns: 'high', 'medium', or 'low'
    """
    pred_text = diff['pred_text']
    ref_text = diff['ref_text']
    
    # Single character substitution - check specific pairs
    if len(pred_text) == 1 and len(ref_text) == 1:
        pair = (pred_text, ref_text)
        
        # TOP PRIORITY - specific important pairs
        if pair in TOP_PRIORITY_PAIRS:
            return 'high'
        
        # Case-only difference
        if pred_text.lower() == ref_text.lower():
            return 'low'
        
        # Punctuation/spacing
        if pred_text in LOW_PRIORITY_CHARS or ref_text in LOW_PRIORITY_CHARS:
            return 'low'
        
        # Other diacritic errors (medium priority)
        if pred_text in DIACRITIC_CHARS or ref_text in DIACRITIC_CHARS:
            return 'medium'
        
        # Other vowel errors (medium priority)
        if pred_text in VOWELS and ref_text in VOWELS:
            return 'medium'
        
        # Consonant errors (medium priority)
        return 'medium'
    
    # Multi-character diffs
    # Check if involves any diacritic chars
    has_diacritic = any(c in DIACRITIC_CHARS for c in pred_text + ref_text)
    if has_diacritic:
        return 'medium'
    
    # Check if purely punctuation/spacing
    pred_content = ''.join(c for c in pred_text if c not in LOW_PRIORITY_CHARS)
    ref_content = ''.join(c for c in ref_text if c not in LOW_PRIORITY_CHARS)
    if not pred_content and not ref_content:
        return 'low'
    
    return 'medium'

print('Priority classification defined!')
print(f'Top priority pairs: {len(TOP_PRIORITY_PAIRS)}')
print(f'Diacritic chars: {len(DIACRITIC_CHARS)}')

In [ ]:
# =============================================================================
# FIND DIFFERENCES BETWEEN STRINGS
# =============================================================================

def get_string_diffs(pred, ref):
    """
    Find differences between pred and ref at character level.
    Returns list of diffs with positions and text.
    """
    matcher = SequenceMatcher(None, pred, ref)
    diffs = []
    
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != 'equal':
            pred_text = pred[i1:i2]
            ref_text = ref[j1:j2]
            diffs.append({
                'pred_start': i1,
                'pred_end': i2,
                'ref_start': j1,
                'ref_end': j2,
                'pred_text': pred_text,
                'ref_text': ref_text,
                'tag': tag
            })
    
    return diffs


def is_punctuation_only_diff(pred_text, ref_text):
    """Check if difference is only punctuation (no content change)."""
    punct = r'[.,;:!?()\[\]"\'\'\-–—]'
    p_clean = re.sub(punct, '', pred_text).strip().lower()
    r_clean = re.sub(punct, '', ref_text).strip().lower()
    p_clean = re.sub(r'\s+', '', p_clean)
    r_clean = re.sub(r'\s+', '', r_clean)
    return p_clean == r_clean


def is_izafet_diff(pred_text, ref_text):
    """Check if difference involves izafet hyphen."""
    pattern = r'-\s*[a-zçşğöüı]'
    return bool(re.search(pattern, pred_text) or re.search(pattern, ref_text))


def filter_diffs(diffs):
    """
    Filter diffs: keep content errors and izafet, exclude punctuation-only.
    Also adds priority classification to each diff.
    """
    filtered = []
    excluded = 0
    
    for diff in diffs:
        if is_izafet_diff(diff['pred_text'], diff['ref_text']):
            diff['reason'] = 'izafet'
            diff['priority'] = classify_diff_priority(diff)
            filtered.append(diff)
        elif is_punctuation_only_diff(diff['pred_text'], diff['ref_text']):
            excluded += 1
        else:
            diff['reason'] = 'content'
            diff['priority'] = classify_diff_priority(diff)
            filtered.append(diff)
    
    return filtered, excluded


print('Diff functions defined!')

In [ ]:
# =============================================================================
# STRING-LEVEL SYNTHETIC GENERATION WITH WEIGHTED PRIORITY SAMPLING
# =============================================================================

def apply_corrections(pred, diffs_to_fix):
    """
    Apply selected corrections to pred string.
    diffs_to_fix should be sorted by pred_start (descending) to avoid offset issues.
    """
    result = pred
    for diff in sorted(diffs_to_fix, key=lambda x: x['pred_start'], reverse=True):
        result = result[:diff['pred_start']] + diff['ref_text'] + result[diff['pred_end']:]
    return result


def weighted_sample(items, weights, k):
    """
    Sample k items without replacement, weighted by weights.
    Higher weight = more likely to be selected.
    """
    if k >= len(items):
        return items[:]
    
    items = list(items)
    weights = list(weights)
    selected = []
    
    for _ in range(k):
        total = sum(weights)
        if total == 0:
            break
        r = random.random() * total
        cumsum = 0
        for i, w in enumerate(weights):
            cumsum += w
            if r <= cumsum:
                selected.append(items[i])
                items.pop(i)
                weights.pop(i)
                break
    
    return selected


def count_priority_remaining(diffs, diffs_fixed):
    """Count how many high/medium/low priority errors remain after fixing."""
    fixed_set = set(id(d) for d in diffs_fixed)
    remaining = [d for d in diffs if id(d) not in fixed_set]
    
    counts = {'high': 0, 'medium': 0, 'low': 0}
    for d in remaining:
        counts[d.get('priority', 'medium')] += 1
    
    return counts


def generate_variants_random_weighted(pred, ref, max_variants=2):
    """
    Generate synthetic training pairs using WEIGHTED random selection.
    
    Key difference: Uses PRIORITY_FIX_WEIGHTS to preferentially fix low-priority
    errors, keeping high-priority errors (diacritics, vowels) more often.
    
    Returns list of (synthetic_input, target, pair_type, n_remaining, n_fixed, 
                     fixed_indices, high_priority_remaining)
    """
    all_diffs = get_string_diffs(pred, ref)
    filtered_diffs, _ = filter_diffs(all_diffs)
    
    n_errors = len(filtered_diffs)
    
    # Count initial priorities
    init_counts = {'high': 0, 'medium': 0, 'low': 0}
    for d in filtered_diffs:
        init_counts[d.get('priority', 'medium')] += 1
    
    # Original pair (no fixes)
    pairs = [(pred, ref, 'original', n_errors, 0, [], init_counts['high'])]
    
    if n_errors < 2:
        return pairs
    
    n_variants = min(max_variants, n_errors - 1)
    
    for v in range(n_variants):
        n_to_fix = random.randint(1, n_errors - 1)
        
        # Build fix weights: higher weight = more likely to FIX
        # We want to fix LOW priority errors more, keep HIGH priority unfixed
        fix_weights = [PRIORITY_FIX_WEIGHTS[d.get('priority', 'medium')] for d in filtered_diffs]
        
        # Weighted sample which diffs to fix
        diffs_to_fix = weighted_sample(filtered_diffs, fix_weights, n_to_fix)
        fixed_indices = sorted([filtered_diffs.index(d) for d in diffs_to_fix])
        
        synthetic_input = apply_corrections(pred, diffs_to_fix)
        
        # Count remaining priorities
        priority_counts = count_priority_remaining(filtered_diffs, diffs_to_fix)
        
        pairs.append((
            synthetic_input, ref, f'random_{v+1}',
            n_errors - n_to_fix, n_to_fix, fixed_indices,
            priority_counts['high']
        ))
    
    return pairs


print('Weighted generator functions defined!')
print(f'Fix weights: {PRIORITY_FIX_WEIGHTS}')

In [ ]:
# =============================================================================
# GENERATE TRAINING DATA WITH PRIORITY-WEIGHTED SAMPLING
# =============================================================================

def generate_train_data_weighted(max_variants=2):
    """
    Generate synthetic training data using WEIGHTED priority sampling.
    High-priority errors (diacritics, vowels) are kept more often.
    """
    all_records = []
    
    for tagged_idx in train_indices:
        if tagged_idx not in df_tagged.index:
            continue
        
        row = df_tagged.loc[tagged_idx]
        pred = row['Better_Pred']
        ref = row['Better_Ref']
        book = row.get('Book', 'unknown')
        page_num = row.get('PageNum', 0)
        
        # Skip if pred or ref is missing
        if pd.isna(pred) or pd.isna(ref):
            continue
        
        # Generate weighted random pairs
        pairs = generate_variants_random_weighted(pred, ref, max_variants)
        
        for inp, tgt, ptype, remaining, fixed, fixed_idx, high_remaining in pairs:
            record = {
                'Better_Pred': pred,
                'Better_Ref': ref,
                'Synthetic_Input': inp,
                'Book': book,
                'PageNum': page_num,
                'pair_type': ptype,
                'errors_remaining': remaining,
                'errors_fixed': fixed,
                'fixed_indices': str(fixed_idx),
                'original_idx': tagged_idx,
                'is_synthetic': ptype != 'original',
                'high_priority_remaining': high_remaining,
            }
            all_records.append(record)
    
    return pd.DataFrame(all_records)


print('='*70)
print('WEIGHTED RANDOM APPROACH (priority-focused)')
print('='*70)
df_train_random = generate_train_data_weighted(max_variants=2)
n_orig = (df_train_random['pair_type'] == 'original').sum()
n_synth = (df_train_random['is_synthetic']).sum()
print(f'Original: {n_orig}, Synthetic: {n_synth}, Total: {len(df_train_random)}')
print(f'Augmentation: {len(df_train_random)/n_orig:.2f}x')

# Show priority distribution in synthetic rows
synth_rows = df_train_random[df_train_random['is_synthetic']]
print(f'\nHigh-priority errors remaining in synthetic rows:')
print(f'  Mean: {synth_rows["high_priority_remaining"].mean():.2f}')
print(f'  Max: {synth_rows["high_priority_remaining"].max()}')
print(f'  Rows with high_priority > 0: {(synth_rows["high_priority_remaining"] > 0).sum()}')

In [ ]:
# =============================================================================
# CREATE COMPLETE DATAFRAME (RANDOM ONLY)
# =============================================================================

def create_complete(df_train):
    """
    Create complete DataFrame with train/dev/test.
    
    - Train rows get negative indices (-1, -2, ...)
    - Dev/test rows KEEP their original df_tagged indices
    """
    
    # Create dev/test from df_tagged using Better columns
    def create_eval_df(indices):
        rows = []
        for idx in indices:
            if idx not in df_tagged.index:
                continue
            row = df_tagged.loc[idx]
            
            # Count high priority errors in eval data
            pred = row['Better_Pred']
            ref = row['Better_Ref']
            high_count = 0
            if not pd.isna(pred) and not pd.isna(ref):
                diffs = get_string_diffs(pred, ref)
                filtered, _ = filter_diffs(diffs)
                high_count = sum(1 for d in filtered if d.get('priority') == 'high')
            
            rows.append({
                'Better_Pred': pred,
                'Better_Ref': ref,
                'Synthetic_Input': pred,  # Input = pred for eval
                'Book': row.get('Book', 'unknown'),
                'PageNum': row.get('PageNum', 0),
                'pair_type': 'original',
                'errors_remaining': -1,
                'errors_fixed': 0,
                'fixed_indices': '[]',
                'original_idx': idx,
                'is_synthetic': False,
                'high_priority_remaining': high_count,
                '_index': idx,
            })
        df = pd.DataFrame(rows)
        df.index = df['_index']
        df = df.drop('_index', axis=1)
        return df
    
    df_dev = create_eval_df(dev_indices)
    df_test = create_eval_df(test_indices)
    
    # Give train negative indices (so they don't conflict)
    df_train = df_train.reset_index(drop=True)
    df_train.index = [-1 - i for i in range(len(df_train))]
    
    # Concatenate: train (negative) + dev (original) + test (original)
    df_complete = pd.concat([df_train, df_dev, df_test], axis=0)
    
    print(f'Complete DataFrame: {len(df_complete)} rows')
    print(f'  Train: {len(df_train)} (indices {df_train.index.min()} to {df_train.index.max()})')
    print(f'  Dev: {len(df_dev)} (original indices)')
    print(f'  Test: {len(df_test)} (original indices)')
    
    return df_complete


print('Creating complete dataframe...')
print('\nWEIGHTED RANDOM:')
df_complete_random = create_complete(df_train_random)

In [ ]:
# =============================================================================
# VERIFY ALIGNMENT AND PRIORITY DISTRIBUTION
# =============================================================================

print('='*70)
print('VERIFICATION: Check alignment with df_synthetic_random.pkl')
print('='*70)

# Load the original random for comparison
df_random_orig = pd.read_pickle('df_synthetic_random.pkl')

# Check that test indices match
sample_test_indices = sorted(list(test_indices))[:5]
print(f'\nSample test indices: {sample_test_indices}')

print(f'\n{"Index":<10} {"Random orig_idx":<18} {"Better orig_idx":<18} {"Match?":<8}')
print('-'*60)

all_match = True
for idx in sample_test_indices:
    if idx in df_random_orig.index and idx in df_complete_random.index:
        r_oid = df_random_orig.loc[idx]['original_idx']
        b_oid = df_complete_random.loc[idx]['original_idx']
        match = '✓' if r_oid == b_oid else '✗'
        if r_oid != b_oid:
            all_match = False
        print(f'{idx:<10} {r_oid:<18} {b_oid:<18} {match:<8}')
    else:
        print(f'{idx:<10} Missing in one or both dataframes')
        all_match = False

print(f'\nAll test indices aligned: {"YES ✓" if all_match else "NO ✗"}')

# Priority distribution analysis
print('\n' + '='*70)
print('PRIORITY DISTRIBUTION IN TRAINING DATA')
print('='*70)

train_rows = df_complete_random[df_complete_random.index < 0]
orig_rows = train_rows[train_rows['pair_type'] == 'original']
synth_rows = train_rows[train_rows['is_synthetic']]

print(f'\nOriginal rows:')
print(f'  Total: {len(orig_rows)}')
print(f'  With high-priority errors: {(orig_rows["high_priority_remaining"] > 0).sum()}')
print(f'  Mean high-priority per row: {orig_rows["high_priority_remaining"].mean():.2f}')

print(f'\nSynthetic rows:')
print(f'  Total: {len(synth_rows)}')
print(f'  With high-priority errors: {(synth_rows["high_priority_remaining"] > 0).sum()}')
print(f'  Mean high-priority per row: {synth_rows["high_priority_remaining"].mean():.2f}')

# Show the effect of weighted sampling
print(f'\n--- Effect of weighted sampling ---')
print(f'If random: synthetic high_priority should be ~same as original')
print(f'With weighting: synthetic should KEEP more high-priority errors')
ratio = synth_rows["high_priority_remaining"].mean() / (orig_rows["high_priority_remaining"].mean() + 0.001)
print(f'Ratio (synth/orig): {ratio:.2f}x')

In [ ]:
# =============================================================================
# SAVE
# =============================================================================

df_complete_random.to_pickle('df_synthetic_better_random.pkl')

print('Saved:')
print(f'  df_synthetic_better_random.pkl: {len(df_complete_random)} rows')
print(f'\nColumns: {list(df_complete_random.columns)}')

print('\n' + '='*70)
print('OUTPUT FILE CONTAINS PRIORITY-WEIGHTED SYNTHETIC DATA')
print('='*70)
print('''
Key features:
- high_priority_remaining column tracks diacritic/vowel errors per row
- Weighted sampling keeps high-priority errors unfixed more often
- Low-priority errors (case, punct) are fixed more often
- Same index structure as original df_synthetic_random.pkl
''')

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print('='*70)
print('PRIORITY-WEIGHTED SYNTHETIC PAIR GENERATION')
print('='*70)
print('''
ERROR PRIORITY LEVELS:
- HIGH: Specific pairs (â↔a, î↔i, i↔ı, e↔a, e↔i) - kept more often
- MEDIUM: Other diacritics, vowels, consonants - neutral
- LOW: Case changes, punctuation, spacing - fixed more often

WEIGHTS FOR FIXING:
- High priority: 0.3 (lowest relative weight, so least likely to be picked for fixing)
- Medium priority: 1.0 (baseline)
- Low priority: 3.0 (3x the weight of medium, so most likely to be picked for fixing)
(Weights are relative: at each draw, an error is picked with probability
 its weight divided by the total weight of the errors not yet picked.)

OUTPUT:
- df_synthetic_better_random.pkl with weighted sampling
- New column: high_priority_remaining (count of high-priority errors)

USE CASE:
Train error correction model to focus on diacritic/vowel errors
rather than trivial case/punctuation fixes.
''')